# 从零开始训练一个微型 GPT 模型

本 Notebook 将带你从零实现一个 **微型 GPT（Generative Pre-trained Transformer）模型**，包括：

1. **数据准备** — 字符级 Tokenizer 与训练数据生成
2. **模型架构** — 完整的 Transformer Decoder（多头自注意力 + 前馈网络）
3. **训练过程** — 自回归语言建模训练
4. **文本生成** — 使用训练好的模型生成新文本
5. **可视化** — 训练损失曲线

这是一个纯学习目的的项目，模型非常小（约 **0.8M 参数**），在 CPU 上几分钟内即可完成训练。

## 1. 导入依赖

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import time

print(f"PyTorch 版本: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

## 2. 准备训练数据

我们使用一段中英文混合的文本作为训练语料。对于微型模型，使用 **字符级 Tokenizer** 最为直观——每个字符对应一个 Token。

在实际的 GPT 中，会使用 BPE（Byte-Pair Encoding）等子词分词器，但字符级分词更适合教学演示。

In [ ]:
text = """
The quick brown fox jumps over the lazy dog.
A journey of a thousand miles begins with a single step.
To be or not to be, that is the question.
All that glitters is not gold.
Knowledge is power, and power is responsibility.
In the middle of difficulty lies opportunity.
The only way to do great work is to love what you do.
Life is what happens when you are busy making other plans.
The future belongs to those who believe in the beauty of their dreams.
It does not matter how slowly you go as long as you do not stop.
Success is not final, failure is not fatal, it is the courage to continue that counts.
The best time to plant a tree was twenty years ago, the second best time is now.
Do not go where the path may lead, go instead where there is no path and leave a trail.
In three words I can sum up everything I have learned about life: it goes on.
The greatest glory in living lies not in never falling, but in rising every time we fall.
It is during our darkest moments that we must focus to see the light.
The purpose of our lives is to be happy.
Life is really simple, but we insist on making it complicated.
The way to get started is to quit talking and begin doing.
If life were predictable it would cease to be life, and be without flavor.
Spread love everywhere you go. Let no one ever come to you without leaving happier.
When you reach the end of your rope, tie a knot in it and hang on.
Always remember that you are absolutely unique. Just like everyone else.
The only impossible journey is the one you never begin.
Tell me and I forget. Teach me and I remember. Involve me and I learn.
The best and most beautiful things in the world cannot be seen or even touched.
It is only with the heart that one can see rightly, what is essential is invisible to the eye.
You must be the change you wish to see in the world.
Not how long, but how well you have lived is the main thing.
If you look at what you have in life, you will always have more.
If you look at what you do not have in life, you will never have enough.
Life itself is the most wonderful fairy tale.
Do not let making a living prevent you from making a life.
Go confidently in the direction of your dreams. Live the life you have imagined.
The real test of a good teacher is whether the students want to keep learning.
A good model learns patterns from data and generalizes to new situations.
Transformers use attention mechanisms to process sequences in parallel.
Neural networks are universal function approximators given enough capacity.
Deep learning has revolutionized natural language processing and computer vision.
The attention mechanism allows the model to focus on relevant parts of the input.
Gradient descent is the workhorse of modern machine learning optimization.
Overfitting occurs when a model memorizes training data instead of learning patterns.
Regularization techniques help prevent overfitting and improve generalization.
The loss function measures how far the model predictions are from the true values.
Backpropagation computes gradients efficiently using the chain rule of calculus.
""".strip()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"语料长度: {len(text)} 字符")
print(f"词汇表大小: {vocab_size}")
print(f"词汇表: {''.join(chars)}")
print(f"\n编码示例: 'The quick' -> {encode('The quick')}")
print(f"解码示例: {encode('The quick')} -> '{decode(encode('The quick'))}'")

## 3. 构建数据集

将文本转换为训练数据：每个样本是一个长度为 `block_size` 的序列，目标是预测下一个 Token。

```
输入:  [T, h, e, ' ', q]
目标:  [h, e, ' ', q, u]    (向右移一位)
```

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(f"数据张量形状: {data.shape}")

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"训练集: {len(train_data)} tokens, 验证集: {len(val_data)} tokens")

block_size = 64
batch_size = 32

def get_batch(split):
    """随机采样一个 batch 的训练数据"""
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print(f"\n输入批次形状: {xb.shape}  (batch_size × block_size)")
print(f"目标批次形状: {yb.shape}")
print(f"\n示例 — 输入: '{decode(xb[0].tolist())[:40]}...'")
print(f"示例 — 目标: '{decode(yb[0].tolist())[:40]}...'")

## 4. 定义模型超参数

以下是我们微型 GPT 的核心参数。相比真正的 GPT-2（1.5B 参数），这个模型非常小，但架构完全一致。

| 参数 | 微型 GPT | GPT-2 Small | GPT-3 |
|------|----------|-------------|-------|
| 嵌入维度 (d_model) | 64 | 768 | 12288 |
| 注意力头数 | 4 | 12 | 96 |
| 层数 | 4 | 12 | 96 |
| 总参数量 | ~0.8M | 124M | 175B |

In [ ]:
n_embd = 64       # 嵌入维度
n_head = 4        # 注意力头数
n_layer = 4       # Transformer 层数
dropout = 0.1     # Dropout 比率
learning_rate = 3e-4

## 5. 构建 GPT 模型

### 5.1 自注意力机制 (Self-Attention)

自注意力是 Transformer 的核心。对于输入序列中的每个位置，它计算与所有其他位置的关联程度（注意力权重），然后加权聚合信息。

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

在 GPT 中使用 **因果掩码 (Causal Mask)**，确保位置 $i$ 只能注意到位置 $\leq i$ 的信息（不能「偷看未来」）。

In [ ]:
class CausalSelfAttention(nn.Module):
    """多头因果自注意力"""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.qkv_proj = nn.Linear(n_embd, 3 * n_embd)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        return y

print("✓ CausalSelfAttention 定义完成")

### 5.2 Transformer Block

每个 Transformer Block 由以下部分组成：
1. **LayerNorm** → **多头自注意力** → 残差连接
2. **LayerNorm** → **前馈网络 (FFN)** → 残差连接

GPT 使用 **Pre-Norm** 架构（先 Norm 后 Attention），这比 Post-Norm 训练更稳定。

In [ ]:
class TransformerBlock(nn.Module):
    """一个 Transformer Decoder Block"""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # 残差连接 + 自注意力
        x = x + self.ffn(self.ln2(x))    # 残差连接 + 前馈网络
        return x

print("✓ TransformerBlock 定义完成")

### 5.3 完整的 MicroGPT 模型

完整模型由以下组件组成：

```
输入 Token IDs
    ↓
Token Embedding + Positional Embedding
    ↓
N × Transformer Block
    ↓
LayerNorm
    ↓
Linear (输出 logits，预测下一个 Token 的概率分布)
```

In [ ]:
class MicroGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size

        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.Sequential(
            *[TransformerBlock(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, f"序列长度 {T} 超过 block_size {self.block_size}"

        tok_emb = self.token_emb(idx)                          # (B, T, n_embd)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))  # (T, n_embd)
        x = self.drop(tok_emb + pos_emb)                       # (B, T, n_embd)

        x = self.blocks(x)                                     # (B, T, n_embd)
        x = self.ln_f(x)                                       # (B, T, n_embd)
        logits = self.lm_head(x)                               # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """自回归生成：逐个 Token 生成"""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


model = MicroGPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"✓ MicroGPT 模型创建完成")
print(f"  总参数量: {n_params:,} ({n_params/1e6:.2f}M)")
print(f"  嵌入维度: {n_embd}")
print(f"  注意力头: {n_head}")
print(f"  层数:     {n_layer}")
print(f"  上下文窗口: {block_size}")

### 训练前生成效果（随机输出）

在训练之前，模型权重是随机初始化的，所以生成的文本完全是乱码。训练后会看到巨大改善。

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("训练前生成（随机乱码）:")
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))
print("---")

## 6. 训练模型

使用 **AdamW 优化器**进行训练。AdamW 是 Adam 的改进版本，修正了权重衰减的实现方式，是训练 Transformer 的标准选择。

训练过程：
1. 随机采样一个 batch
2. 前向传播计算 loss
3. 反向传播计算梯度
4. 更新参数
5. 重复

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

max_iters = 3000
eval_interval = 300
eval_iters = 50

@torch.no_grad()
def estimate_loss():
    """估算训练集和验证集的平均损失"""
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

train_losses = []
val_losses = []
steps = []

print("开始训练...")
print(f"总迭代次数: {max_iters}, 每 {eval_interval} 步评估一次")
print("-" * 60)

start_time = time.time()

for iter_num in range(max_iters):
    if iter_num % eval_interval == 0 or iter_num == max_iters - 1:
        losses = estimate_loss()
        elapsed = time.time() - start_time
        print(f"步骤 {iter_num:>5d} | 训练损失: {losses['train']:.4f} | 验证损失: {losses['val']:.4f} | 用时: {elapsed:.1f}s")
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        steps.append(iter_num)

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

total_time = time.time() - start_time
print("-" * 60)
print(f"训练完成! 总用时: {total_time:.1f}s")

## 7. 可视化训练过程

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(steps, train_losses, 'b-o', label='Train Loss', markersize=4)
plt.plot(steps, val_losses, 'r-o', label='Val Loss', markersize=4)
plt.xlabel('Training Steps')
plt.ylabel('Cross-Entropy Loss')
plt.title('MicroGPT Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print("损失曲线已保存为 training_loss.png")

## 8. 文本生成

模型训练完成后，我们可以用它来生成文本。通过调节 **temperature** 参数来控制生成的多样性：
- `temperature < 1.0`：更保守、更确定的输出
- `temperature = 1.0`：标准采样
- `temperature > 1.0`：更随机、更多样的输出

`top_k` 参数限制每步只从概率最高的 k 个 Token 中采样。

In [ ]:
model.eval()

prompts = ["The ", "In t", "Life", "Deep"]

print("=" * 60)
print("文本生成演示")
print("=" * 60)

for prompt in prompts:
    context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    generated = decode(model.generate(context, max_new_tokens=150, temperature=0.8, top_k=10)[0].tolist())
    print(f"\n--- 提示词: '{prompt}' ---")
    print(generated)

print("\n" + "=" * 60)

## 9. 不同 Temperature 的效果对比

观察 temperature 参数如何影响生成文本的多样性。

In [ ]:
prompt = "The "
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

temperatures = [0.5, 0.8, 1.0, 1.5]

print(f"提示词: '{prompt}'")
print("=" * 60)

for temp in temperatures:
    generated = decode(model.generate(context, max_new_tokens=100, temperature=temp, top_k=15)[0].tolist())
    print(f"\nTemperature = {temp}:")
    print(f"  {generated}")

print("\n" + "=" * 60)

## 10. 模型内部分析

让我们看看模型学到了什么——可视化注意力权重和 Token 嵌入。

In [ ]:
with torch.no_grad():
    sample_text = "The quick brown"
    sample_ids = torch.tensor([encode(sample_text)], dtype=torch.long, device=device)
    T = sample_ids.shape[1]

    tok_emb = model.token_emb(sample_ids)
    pos_emb = model.pos_emb(torch.arange(T, device=device))
    x = model.drop(tok_emb + pos_emb)

    first_block = model.blocks[0]
    ln_out = first_block.ln1(x)

    B, T_len, C = ln_out.shape
    qkv = first_block.attn.qkv_proj(ln_out)
    q, k, v = qkv.split(C, dim=2)
    head_dim = C // n_head
    q = q.view(B, T_len, n_head, head_dim).transpose(1, 2)
    k = k.view(B, T_len, n_head, head_dim).transpose(1, 2)

    att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_dim))
    mask = first_block.attn.mask[:, :, :T_len, :T_len]
    att = att.masked_fill(mask == 0, float('-inf'))
    att = F.softmax(att, dim=-1)

fig, axes = plt.subplots(1, n_head, figsize=(16, 4))
char_labels = list(sample_text)

for h in range(n_head):
    ax = axes[h]
    im = ax.imshow(att[0, h].cpu().numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {h+1}', fontsize=11)
    ax.set_xticks(range(len(char_labels)))
    ax.set_yticks(range(len(char_labels)))
    ax.set_xticklabels(char_labels, fontsize=9)
    ax.set_yticklabels(char_labels, fontsize=9)

fig.suptitle('Attention Weights (Layer 1)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('attention_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print("注意力权重图已保存为 attention_weights.png")

## 总结

在这个 Notebook 中，我们从零实现了一个完整的 GPT 模型，包括：

| 组件 | 说明 |
|------|------|
| **字符级 Tokenizer** | 将文本转换为 Token 序列 |
| **Token + 位置嵌入** | 将 Token 映射到连续向量空间 |
| **多头因果自注意力** | 捕获序列中的依赖关系 |
| **前馈网络 (FFN)** | 对每个位置独立进行非线性变换 |
| **残差连接 + LayerNorm** | 稳定深层网络训练 |
| **自回归生成** | 逐个 Token 生成新文本 |

### 进一步学习方向

- 增大模型规模和训练数据，观察生成质量变化
- 尝试 BPE 分词器替代字符级分词
- 实现学习率预热 (warmup) 和余弦退火调度
- 添加 KV-Cache 加速推理
- 阅读 GPT-2 原始论文: [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- 学习本课程的其他 Lesson，了解如何将 LLM 用于构建 AI Agent